# Session 9.5: Model Monitoring

**Duration:** 120 minutes

## Learning Objectives
By the end of this session, you will be able to:
1. Understand why machine learning models degrade over time
2. Identify data drift and concept drift
3. Implement logging for ML predictions
4. Track key monitoring metrics (request volume, prediction distribution, confidence)
5. Build a real-time monitoring dashboard with Streamlit
6. Create alerts for anomalies and performance degradation
7. Simulate model drift and detect it

## Prerequisites
- Completed Sessions 9.1-9.4 (especially Cloud Deployment)
- Deployed API from Session 9.4
- Basic understanding of statistics

---
## Part 1: Why Models Degrade (15 minutes)

### The Problem: Models Don't Last Forever

You've trained a model, it performs great on test data, and you deploy it. Success! But then...

- **Week 1**: 92% accuracy
- **Month 1**: 88% accuracy
- **Month 3**: 79% accuracy
- **Month 6**: 65% accuracy

**What happened?** The world changed, but your model didn't.

### Types of Model Degradation

#### 1. Data Drift (Covariate Shift)

**Definition:** The distribution of input features changes over time.

**Example:**
- **Training data (2019):** Average customer age = 35, income = $50k
- **Production data (2024):** Average customer age = 42, income = $65k

The relationship between features and target may still be the same, but the features themselves have shifted.

**Real-world causes:**
- Demographic changes
- Seasonal variations
- Economic shifts
- New user segments

#### 2. Concept Drift

**Definition:** The relationship between features and target changes over time.

**Example:**
- **Before COVID-19:** Working from home = indicator of low job satisfaction
- **After COVID-19:** Working from home = normal, not predictive

The features are the same, but their meaning has changed.

**Real-world causes:**
- Changing user behavior
- Market disruptions
- Regulatory changes
- Competitor actions

#### 3. Label Drift

**Definition:** The distribution of target values changes.

**Example:**
- **Training data:** 90% negative reviews, 10% positive
- **Production data:** 60% negative reviews, 40% positive

### Real-World Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Simulate data drift
np.random.seed(42)

# Training data (2019): Younger customers
X_train = np.random.normal(loc=35, scale=10, size=(1000, 1))  # Age
y_train = (X_train[:, 0] > 40).astype(int)  # Buy if age > 40

# Production data shifts over time
months = 12
accuracies = []
mean_ages = []

# Train model on original data
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

for month in range(months):
    # Data drifts: customers get older over time
    age_shift = month * 0.5
    X_prod = np.random.normal(loc=35 + age_shift, scale=10, size=(500, 1))
    y_prod = (X_prod[:, 0] > 40).astype(int)
    
    # Evaluate
    y_pred = model.predict(X_prod)
    acc = accuracy_score(y_prod, y_pred)
    
    accuracies.append(acc)
    mean_ages.append(X_prod.mean())

# Visualize degradation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy over time
ax1.plot(range(1, months + 1), accuracies, marker='o', linewidth=2, markersize=8)
ax1.axhline(y=0.85, color='r', linestyle='--', label='Acceptable threshold')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Performance Degradation Over Time', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Feature drift
ax2.plot(range(1, months + 1), mean_ages, marker='s', linewidth=2, markersize=8, color='orange')
ax2.axhline(y=35, color='g', linestyle='--', label='Training data mean')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Mean Age', fontsize=12)
ax2.set_title('Data Drift: Mean Age Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial accuracy: {accuracies[0]:.3f}")
print(f"Final accuracy: {accuracies[-1]:.3f}")
print(f"Degradation: {(accuracies[0] - accuracies[-1]) * 100:.1f}%")

### Why Monitoring is Critical

Without monitoring, you won't know when your model starts failing. This leads to:

- **Silent failures**: Bad predictions without anyone noticing
- **User dissatisfaction**: Poor experience with your application
- **Business losses**: Wrong decisions based on bad predictions
- **Reputation damage**: Users lose trust in your system

**The solution:** Continuous monitoring and timely retraining!

---
## Part 2: Monitoring Metrics (15 minutes)

### What to Monitor

#### 1. Request Volume Metrics

In [ ]:
# Example request volume tracking
from datetime import datetime, timedelta

# Simulate request logs
np.random.seed(42)
dates = pd.date_range(end=datetime.now(), periods=30, freq='D')

# Normal traffic with some anomalies
requests_per_day = []
for i, date in enumerate(dates):
    if i == 15:  # Simulate outage
        count = 50
    elif i == 20:  # Simulate spike
        count = 2500
    else:
        count = np.random.normal(1000, 150)
    requests_per_day.append(max(0, int(count)))

df_requests = pd.DataFrame({
    'date': dates,
    'requests': requests_per_day
})

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(df_requests['date'], df_requests['requests'], marker='o', linewidth=2)
plt.axhline(y=df_requests['requests'].mean(), color='g', linestyle='--', 
            label=f'Mean: {df_requests["requests"].mean():.0f}')
plt.axhline(y=df_requests['requests'].mean() - 2*df_requests['requests'].std(), 
            color='r', linestyle='--', label='Alert threshold (low)')
plt.axhline(y=df_requests['requests'].mean() + 2*df_requests['requests'].std(), 
            color='r', linestyle='--', label='Alert threshold (high)')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Requests per Day', fontsize=12)
plt.title('API Request Volume Monitoring', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Key Metrics:")
print(f"Average requests/day: {df_requests['requests'].mean():.0f}")
print(f"Std deviation: {df_requests['requests'].std():.0f}")
print(f"Min: {df_requests['requests'].min()} (Day {df_requests['requests'].idxmin()})")
print(f"Max: {df_requests['requests'].max()} (Day {df_requests['requests'].idxmax()})")

#### 2. Response Time Metrics

In [ ]:
# Simulate response times
response_times = np.random.gamma(2, 50, size=1000)  # Most requests fast, some slow

plt.figure(figsize=(14, 5))
plt.hist(response_times, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=np.percentile(response_times, 95), color='r', linestyle='--',
            label=f'P95: {np.percentile(response_times, 95):.0f}ms')
plt.axvline(x=np.percentile(response_times, 99), color='orange', linestyle='--',
            label=f'P99: {np.percentile(response_times, 99):.0f}ms')
plt.xlabel('Response Time (ms)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('API Response Time Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean response time: {response_times.mean():.1f}ms")
print(f"Median (P50): {np.percentile(response_times, 50):.1f}ms")
print(f"P95: {np.percentile(response_times, 95):.1f}ms")
print(f"P99: {np.percentile(response_times, 99):.1f}ms")

#### 3. Prediction Distribution

In [ ]:
# Simulate predictions over time
# Week 1: Normal distribution
predictions_week1 = np.random.normal(100, 15, size=1000)

# Week 4: Distribution has shifted (data drift!)
predictions_week4 = np.random.normal(115, 20, size=1000)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Week 1
ax1.hist(predictions_week1, bins=30, alpha=0.7, color='blue', edgecolor='black')
ax1.axvline(x=predictions_week1.mean(), color='r', linestyle='--', 
            label=f'Mean: {predictions_week1.mean():.1f}')
ax1.set_xlabel('Predicted Price', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Week 1: Baseline', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Week 4
ax2.hist(predictions_week4, bins=30, alpha=0.7, color='orange', edgecolor='black')
ax2.axvline(x=predictions_week4.mean(), color='r', linestyle='--', 
            label=f'Mean: {predictions_week4.mean():.1f}')
ax2.set_xlabel('Predicted Price', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Week 4: Shifted Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Overlay comparison
ax3.hist(predictions_week1, bins=30, alpha=0.5, color='blue', label='Week 1', edgecolor='black')
ax3.hist(predictions_week4, bins=30, alpha=0.5, color='orange', label='Week 4', edgecolor='black')
ax3.set_xlabel('Predicted Price', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('Comparison: Drift Detected!', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Distribution Shift Analysis:")
print(f"Week 1 mean: {predictions_week1.mean():.2f}")
print(f"Week 4 mean: {predictions_week4.mean():.2f}")
print(f"Shift: {predictions_week4.mean() - predictions_week1.mean():.2f} ({((predictions_week4.mean() / predictions_week1.mean() - 1) * 100):.1f}%)")

#### 4. Model Confidence/Probability Scores

In [ ]:
# For classification: monitor prediction probabilities
# Low confidence = uncertain predictions = possible data drift

# Simulate confidence scores
confidence_normal = np.random.beta(8, 2, size=1000)  # High confidence
confidence_drift = np.random.beta(3, 3, size=1000)   # Lower confidence (uncertain)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(confidence_normal, bins=30, alpha=0.7, color='green', edgecolor='black')
ax1.axvline(x=confidence_normal.mean(), color='r', linestyle='--',
            label=f'Mean: {confidence_normal.mean():.3f}')
ax1.set_xlabel('Confidence Score', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Normal: High Confidence', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.hist(confidence_drift, bins=30, alpha=0.7, color='red', edgecolor='black')
ax2.axvline(x=confidence_drift.mean(), color='orange', linestyle='--',
            label=f'Mean: {confidence_drift.mean():.3f}')
ax2.set_xlabel('Confidence Score', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Drift Detected: Low Confidence', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Normal mean confidence: {confidence_normal.mean():.3f}")
print(f"Drift mean confidence: {confidence_drift.mean():.3f}")
print(f"Drop in confidence: {(1 - confidence_drift.mean()/confidence_normal.mean())*100:.1f}%")

### Summary of Key Metrics

| Metric Category | What to Track | Alert Threshold |
|----------------|---------------|------------------|
| **Volume** | Requests/day, Requests/hour | ±2σ from mean |
| **Performance** | Response time (P50, P95, P99) | P95 > 1000ms |
| **Predictions** | Distribution mean/std, Range | Mean shift > 10% |
| **Confidence** | Average confidence score | Drop > 15% |
| **Features** | Input distribution | KS test p < 0.05 |
| **Errors** | Error rate, Error types | > 5% error rate |

---
## Part 3: Logging Best Practices (20 minutes)

### What to Log

Every prediction should be logged with:

1. **Timestamp**: When the prediction was made
2. **Input features**: What data was used
3. **Prediction**: The model's output
4. **Confidence/probability**: Model's certainty
5. **Model version**: Which version made the prediction
6. **Response time**: How long it took
7. **User ID** (optional): Who requested it
8. **Ground truth** (when available): Actual outcome

### Where to Log

**Options:**
1. **CSV files** - Simple, good for small scale
2. **SQLite database** - Queryable, better than CSV
3. **PostgreSQL/MySQL** - Production scale
4. **Cloud logging** (CloudWatch, Stackdriver) - Managed solution
5. **Specialized ML platforms** (MLflow, Neptune, Weights & Biases)

### Implementing Logging

In [ ]:
import sqlite3
import json
from datetime import datetime
import time

class PredictionLogger:
    """Logger for ML predictions."""
    
    def __init__(self, db_path='predictions.db'):
        self.db_path = db_path
        self.init_database()
    
    def init_database(self):
        """Initialize SQLite database with schema."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS predictions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                model_version TEXT NOT NULL,
                input_features TEXT NOT NULL,
                prediction REAL NOT NULL,
                confidence REAL,
                response_time_ms REAL,
                user_id TEXT,
                ground_truth REAL
            )
        ''')
        
        # Create indexes for faster queries
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_timestamp ON predictions(timestamp)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_model_version ON predictions(model_version)')
        
        conn.commit()
        conn.close()
        print(f"Database initialized: {self.db_path}")
    
    def log_prediction(self, model_version, input_features, prediction, 
                      confidence=None, response_time_ms=None, user_id=None, ground_truth=None):
        """Log a single prediction."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            INSERT INTO predictions 
            (timestamp, model_version, input_features, prediction, confidence, 
             response_time_ms, user_id, ground_truth)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            datetime.now().isoformat(),
            model_version,
            json.dumps(input_features),
            float(prediction),
            float(confidence) if confidence else None,
            float(response_time_ms) if response_time_ms else None,
            user_id,
            float(ground_truth) if ground_truth else None
        ))
        
        conn.commit()
        conn.close()
    
    def get_recent_predictions(self, limit=100):
        """Retrieve recent predictions."""
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query(
            f'SELECT * FROM predictions ORDER BY timestamp DESC LIMIT {limit}',
            conn
        )
        conn.close()
        return df
    
    def get_predictions_by_date_range(self, start_date, end_date):
        """Get predictions within date range."""
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query(
            '''SELECT * FROM predictions 
               WHERE timestamp BETWEEN ? AND ?
               ORDER BY timestamp''',
            conn,
            params=(start_date, end_date)
        )
        conn.close()
        return df

# Example usage
logger = PredictionLogger('monitoring_dashboard/predictions.db')

# Simulate logging predictions
print("Logging sample predictions...")
for i in range(10):
    input_data = {
        'day_of_year': np.random.randint(1, 366),
        'day_of_week': np.random.randint(0, 7),
        'trend': i * 10,
        'price_lag_1': 100 + np.random.randn() * 5
    }
    
    prediction = 100 + np.random.randn() * 10
    confidence = np.random.uniform(0.7, 0.95)
    response_time = np.random.uniform(50, 200)
    
    logger.log_prediction(
        model_version='v1.0.0',
        input_features=input_data,
        prediction=prediction,
        confidence=confidence,
        response_time_ms=response_time
    )
    time.sleep(0.1)  # Small delay

print("\nRecent predictions:")
recent = logger.get_recent_predictions(5)
print(recent[['timestamp', 'model_version', 'prediction', 'confidence', 'response_time_ms']].to_string())

### Privacy Considerations

**Important:** Be careful what you log!

**DO:**
- Log aggregate statistics
- Log feature distributions (not individual values for sensitive data)
- Anonymize user IDs
- Encrypt sensitive data

**DON'T:**
- Log personally identifiable information (PII)
- Log credit card numbers, SSNs, passwords
- Log medical information without proper safeguards
- Store logs indefinitely (implement retention policies)

**Best Practice:** Hash or anonymize sensitive features

In [ ]:
import hashlib

def anonymize_user_id(user_id):
    """Hash user ID for privacy."""
    return hashlib.sha256(str(user_id).encode()).hexdigest()[:16]

# Example
original_id = "user_12345"
anonymous_id = anonymize_user_id(original_id)
print(f"Original: {original_id}")
print(f"Anonymized: {anonymous_id}")
print(f"Consistent: {anonymize_user_id(original_id) == anonymous_id}")

---
## Part 4: Alerts and Dashboards (15 minutes)

### When to Alert

Create alerts for:

1. **Volume anomalies**: Requests drop/spike beyond threshold
2. **Performance degradation**: Response time > SLA
3. **Prediction drift**: Distribution shift detected
4. **Low confidence**: Average confidence drops
5. **High error rate**: Errors > 5%
6. **Model failures**: Crashes or timeouts

### Implementing Alerts

In [ ]:
class AlertSystem:
    """Simple alerting system for ML monitoring."""
    
    def __init__(self, logger):
        self.logger = logger
        self.alerts = []
    
    def check_request_volume(self, threshold_std=2):
        """Alert if request volume is anomalous."""
        # Get last 7 days
        end_date = datetime.now()
        start_date = end_date - timedelta(days=7)
        
        df = self.logger.get_predictions_by_date_range(
            start_date.isoformat(),
            end_date.isoformat()
        )
        
        # Group by date
        df['date'] = pd.to_datetime(df['timestamp']).dt.date
        daily_counts = df.groupby('date').size()
        
        if len(daily_counts) < 2:
            return None
        
        # Check today vs historical
        today_count = daily_counts.iloc[-1]
        historical_mean = daily_counts.iloc[:-1].mean()
        historical_std = daily_counts.iloc[:-1].std()
        
        z_score = abs(today_count - historical_mean) / (historical_std + 1e-6)
        
        if z_score > threshold_std:
            alert = {
                'type': 'VOLUME_ANOMALY',
                'severity': 'HIGH' if z_score > 3 else 'MEDIUM',
                'message': f'Request volume anomaly: {today_count} vs avg {historical_mean:.0f} (z={z_score:.1f})',
                'timestamp': datetime.now().isoformat()
            }
            self.alerts.append(alert)
            return alert
        return None
    
    def check_prediction_drift(self, window_days=7, threshold=0.15):
        """Alert if prediction distribution has drifted."""
        end_date = datetime.now()
        recent_start = end_date - timedelta(days=window_days)
        baseline_start = end_date - timedelta(days=window_days*2)
        baseline_end = recent_start
        
        # Get baseline and recent predictions
        df_baseline = self.logger.get_predictions_by_date_range(
            baseline_start.isoformat(),
            baseline_end.isoformat()
        )
        df_recent = self.logger.get_predictions_by_date_range(
            recent_start.isoformat(),
            end_date.isoformat()
        )
        
        if len(df_baseline) < 10 or len(df_recent) < 10:
            return None
        
        # Compare means
        baseline_mean = df_baseline['prediction'].mean()
        recent_mean = df_recent['prediction'].mean()
        
        drift_pct = abs(recent_mean - baseline_mean) / baseline_mean
        
        if drift_pct > threshold:
            alert = {
                'type': 'PREDICTION_DRIFT',
                'severity': 'HIGH' if drift_pct > 0.25 else 'MEDIUM',
                'message': f'Prediction drift detected: {drift_pct*100:.1f}% change (baseline: {baseline_mean:.2f}, recent: {recent_mean:.2f})',
                'timestamp': datetime.now().isoformat()
            }
            self.alerts.append(alert)
            return alert
        return None
    
    def check_low_confidence(self, threshold=0.7):
        """Alert if average confidence is low."""
        # Get recent predictions
        df = self.logger.get_recent_predictions(limit=100)
        
        if len(df) < 10:
            return None
        
        avg_confidence = df['confidence'].mean()
        
        if avg_confidence < threshold:
            alert = {
                'type': 'LOW_CONFIDENCE',
                'severity': 'MEDIUM',
                'message': f'Low average confidence: {avg_confidence:.3f} (threshold: {threshold})',
                'timestamp': datetime.now().isoformat()
            }
            self.alerts.append(alert)
            return alert
        return None
    
    def run_all_checks(self):
        """Run all monitoring checks."""
        print("Running monitoring checks...\n")
        
        checks = [
            ('Request Volume', self.check_request_volume),
            ('Prediction Drift', self.check_prediction_drift),
            ('Confidence Score', self.check_low_confidence)
        ]
        
        new_alerts = []
        for name, check_func in checks:
            try:
                alert = check_func()
                if alert:
                    new_alerts.append(alert)
                    print(f"⚠️  {name}: {alert['message']}")
                else:
                    print(f"✓ {name}: OK")
            except Exception as e:
                print(f"✗ {name}: Error - {e}")
        
        return new_alerts

# Example usage
alert_system = AlertSystem(logger)
alerts = alert_system.run_all_checks()

if alerts:
    print(f"\n{len(alerts)} alert(s) triggered!")
else:
    print("\nNo alerts - system healthy!")

---
## Part 5: Hands-On Monitoring Dashboard (50 minutes)

Now let's build a complete monitoring dashboard using Streamlit!

### Dashboard Features

Our dashboard will show:
1. Total predictions count
2. Predictions per day (line chart)
3. Prediction distribution (histogram)
4. Average confidence over time
5. Response time metrics
6. Active alerts
7. Model version breakdown

The full dashboard code is in `monitoring_dashboard/app.py`, but let's create a simplified version here first:

In [ ]:
# First, let's generate more comprehensive test data
print("Generating test data for dashboard...")

# Simulate 30 days of predictions
np.random.seed(42)
base_date = datetime.now() - timedelta(days=30)

for day in range(30):
    current_date = base_date + timedelta(days=day)
    
    # Variable number of predictions per day
    n_predictions = np.random.randint(50, 200)
    
    for i in range(n_predictions):
        # Add some drift over time
        drift_factor = 1 + (day / 30) * 0.2  # 20% drift over 30 days
        
        input_data = {
            'day_of_year': np.random.randint(1, 366),
            'day_of_week': np.random.randint(0, 7),
            'trend': day * 10,
            'price_lag_1': 100 * drift_factor + np.random.randn() * 5
        }
        
        prediction = 100 * drift_factor + np.random.randn() * 10
        
        # Confidence decreases slightly with drift
        base_confidence = 0.9 - (day / 30) * 0.15
        confidence = base_confidence + np.random.uniform(-0.05, 0.05)
        confidence = np.clip(confidence, 0.5, 0.99)
        
        response_time = np.random.gamma(2, 50)
        
        # Temporarily change timestamp for historical data
        original_now = datetime.now
        datetime.now = lambda: current_date
        
        logger.log_prediction(
            model_version='v1.0.0',
            input_features=input_data,
            prediction=prediction,
            confidence=confidence,
            response_time_ms=response_time
        )
        
        datetime.now = original_now

print(f"Generated test data complete!")
print(f"Total predictions in database: {len(logger.get_recent_predictions(limit=10000))}")

### Visualize Monitoring Data

In [ ]:
# Load all data
end_date = datetime.now()
start_date = end_date - timedelta(days=30)
df_all = logger.get_predictions_by_date_range(start_date.isoformat(), end_date.isoformat())

# Convert timestamp to datetime
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])
df_all['date'] = df_all['timestamp'].dt.date

print(f"Loaded {len(df_all)} predictions")
print(f"Date range: {df_all['date'].min()} to {df_all['date'].max()}")

# Create comprehensive monitoring dashboard
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Predictions per day
ax1 = fig.add_subplot(gs[0, :])
daily_counts = df_all.groupby('date').size()
ax1.plot(daily_counts.index, daily_counts.values, marker='o', linewidth=2, markersize=6)
ax1.axhline(y=daily_counts.mean(), color='r', linestyle='--', label=f'Average: {daily_counts.mean():.0f}')
ax1.fill_between(daily_counts.index, 
                  daily_counts.mean() - daily_counts.std(),
                  daily_counts.mean() + daily_counts.std(),
                  alpha=0.2, color='red', label='±1 std')
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Predictions Count', fontsize=12)
ax1.set_title('Daily Prediction Volume', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. Prediction distribution over time
ax2 = fig.add_subplot(gs[1, 0])
df_all['week'] = df_all['timestamp'].dt.isocalendar().week
weekly_avg = df_all.groupby('week')['prediction'].mean()
ax2.bar(range(len(weekly_avg)), weekly_avg.values, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Week', fontsize=12)
ax2.set_ylabel('Average Prediction', fontsize=12)
ax2.set_title('Average Prediction by Week (Drift Detection)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Confidence over time
ax3 = fig.add_subplot(gs[1, 1])
weekly_confidence = df_all.groupby('week')['confidence'].mean()
ax3.plot(range(len(weekly_confidence)), weekly_confidence.values, 
         marker='o', linewidth=2, markersize=8, color='green')
ax3.axhline(y=0.7, color='r', linestyle='--', label='Alert threshold')
ax3.set_xlabel('Week', fontsize=12)
ax3.set_ylabel('Average Confidence', fontsize=12)
ax3.set_title('Model Confidence Over Time', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Response time distribution
ax4 = fig.add_subplot(gs[2, 0])
ax4.hist(df_all['response_time_ms'], bins=50, alpha=0.7, edgecolor='black', color='purple')
ax4.axvline(x=df_all['response_time_ms'].quantile(0.95), color='r', linestyle='--',
            label=f'P95: {df_all["response_time_ms"].quantile(0.95):.0f}ms')
ax4.set_xlabel('Response Time (ms)', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Response Time Distribution', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 5. Prediction distribution (current)
ax5 = fig.add_subplot(gs[2, 1])
ax5.hist(df_all['prediction'], bins=50, alpha=0.7, edgecolor='black', color='orange')
ax5.axvline(x=df_all['prediction'].mean(), color='r', linestyle='--',
            label=f'Mean: {df_all["prediction"].mean():.2f}')
ax5.axvline(x=df_all['prediction'].median(), color='g', linestyle='--',
            label=f'Median: {df_all["prediction"].median():.2f}')
ax5.set_xlabel('Predicted Value', fontsize=12)
ax5.set_ylabel('Frequency', fontsize=12)
ax5.set_title('Current Prediction Distribution', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

plt.suptitle('ML Model Monitoring Dashboard', fontsize=16, fontweight='bold', y=0.995)
plt.show()

# Print key statistics
print("\n" + "="*60)
print("KEY METRICS SUMMARY")
print("="*60)
print(f"Total Predictions: {len(df_all):,}")
print(f"Date Range: {df_all['date'].min()} to {df_all['date'].max()}")
print(f"\nDaily Volume:")
print(f"  Average: {daily_counts.mean():.0f} predictions/day")
print(f"  Min: {daily_counts.min()} | Max: {daily_counts.max()}")
print(f"\nPredictions:")
print(f"  Mean: {df_all['prediction'].mean():.2f}")
print(f"  Std: {df_all['prediction'].std():.2f}")
print(f"  Range: [{df_all['prediction'].min():.2f}, {df_all['prediction'].max():.2f}]")
print(f"\nConfidence:")
print(f"  Average: {df_all['confidence'].mean():.3f}")
print(f"  Min: {df_all['confidence'].min():.3f}")
print(f"\nResponse Time:")
print(f"  Mean: {df_all['response_time_ms'].mean():.1f}ms")
print(f"  P50: {df_all['response_time_ms'].median():.1f}ms")
print(f"  P95: {df_all['response_time_ms'].quantile(0.95):.1f}ms")
print(f"  P99: {df_all['response_time_ms'].quantile(0.99):.1f}ms")
print("="*60)

### Detecting Drift with Statistical Tests

In [ ]:
from scipy import stats

# Compare first week vs last week
first_week = df_all[df_all['week'] == df_all['week'].min()]['prediction']
last_week = df_all[df_all['week'] == df_all['week'].max()]['prediction']

# Kolmogorov-Smirnov test (tests if distributions are different)
ks_statistic, ks_pvalue = stats.ks_2samp(first_week, last_week)

# T-test (tests if means are different)
t_statistic, t_pvalue = stats.ttest_ind(first_week, last_week)

print("Statistical Drift Detection:")
print(f"\nFirst Week: mean={first_week.mean():.2f}, std={first_week.std():.2f}")
print(f"Last Week: mean={last_week.mean():.2f}, std={last_week.std():.2f}")
print(f"\nKolmogorov-Smirnov Test:")
print(f"  Statistic: {ks_statistic:.4f}")
print(f"  P-value: {ks_pvalue:.4f}")
print(f"  Result: {'DRIFT DETECTED' if ks_pvalue < 0.05 else 'No significant drift'}")
print(f"\nT-Test (mean difference):")
print(f"  Statistic: {t_statistic:.4f}")
print(f"  P-value: {t_pvalue:.4f}")
print(f"  Result: {'MEANS SIGNIFICANTLY DIFFERENT' if t_pvalue < 0.05 else 'No significant difference'}")

# Visualize distributions
plt.figure(figsize=(12, 5))
plt.hist(first_week, bins=30, alpha=0.5, label='First Week', edgecolor='black')
plt.hist(last_week, bins=30, alpha=0.5, label='Last Week', edgecolor='black')
plt.axvline(x=first_week.mean(), color='blue', linestyle='--', label=f'First Week Mean: {first_week.mean():.2f}')
plt.axvline(x=last_week.mean(), color='orange', linestyle='--', label=f'Last Week Mean: {last_week.mean():.2f}')
plt.xlabel('Prediction Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title(f'Distribution Comparison: First vs Last Week (KS p-value: {ks_pvalue:.4f})', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## LLM Prompts for Monitoring

### Prompt 1: Create Monitoring Dashboard

In [ ]:
prompt_monitoring = """
Create a Streamlit monitoring dashboard for a machine learning model with these requirements:

Data Source:
- SQLite database with predictions table
- Columns: timestamp, model_version, prediction, confidence, response_time_ms

Dashboard Features:
1. Overview section:
   - Total predictions count
   - Average confidence
   - Average response time
   - Date range selector

2. Time series charts:
   - Predictions per day (line chart)
   - Average confidence over time
   - Response time (P50, P95, P99)

3. Distribution analysis:
   - Prediction distribution histogram
   - Confidence distribution

4. Alerts section:
   - Show active alerts
   - Low confidence warning
   - Volume anomaly detection

5. Auto-refresh every 30 seconds

Include all necessary imports and make it production-ready.
"""

print(prompt_monitoring)

---
## Summary & Next Steps

### What You Learned

✅ Why models degrade (data drift, concept drift)
✅ Key monitoring metrics (volume, performance, predictions, confidence)
✅ Logging best practices and privacy considerations
✅ Building alerts for anomaly detection
✅ Creating monitoring dashboards with visualization
✅ Statistical tests for drift detection

### Key Takeaways

1. **Monitor Everything**: Log all predictions and track metrics
2. **Detect Drift Early**: Use statistical tests and visual inspection
3. **Set Smart Alerts**: Not too sensitive, not too late
4. **Protect Privacy**: Anonymize sensitive data
5. **Automate Monitoring**: Dashboard should update automatically

### Next Session Preview

**Session 9.6: A/B Testing & Model Updates**
- Deploy multiple model versions simultaneously
- A/B test new vs old models
- Gradual rollout strategies (canary deployment)
- Rollback procedures when things go wrong

### Practice Exercises

1. **Add logging to your deployed API** (Session 9.4)
2. **Create a Streamlit dashboard** using the monitoring_dashboard code
3. **Simulate drift** and verify your alerts trigger
4. **Implement email alerts** using smtplib or SendGrid